# 05 — LSTM Autoencoder Baseline

A non-linear reconstruction baseline. An LSTM encoder compresses each 100-row window
into a latent vector; an LSTM decoder reconstructs the window. Anomalous windows
reconstruct poorly, producing a higher window-mean MSE.

**Inputs**: preprocessed arrays from `data/processed/` (built by NB 02)
**Outputs**: `submissions/baseline_lstm_ae.parquet`, `models/lstm_ae.keras`

Scoring and threshold selection are identical to NB 04 (window-mean MSE,
corrected event-wise F0.5) so the two baselines are directly comparable.

**Sections**
1. Load data
2. Build and train LSTM-AE
3. Score validation and test (window-mean MSE)
4. Threshold-transfer gate
5. Threshold tuning (corrected event-wise F0.5)
6. Validation timeline and per-event analysis
7. Sensitivity — latent_dim and hidden_dim
8. Generate test predictions and save submission
9. Compare to PCA baseline
10. Ensemble with PCA baseline (rank-averaged)

## 0 — Setup & Imports

In [ ]:
import sys, gc, json, time, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sentinel.ml_logic.data    import find_anomaly_segments
from sentinel.ml_logic.metrics import corrected_event_f05
from sentinel.params import (
    RANDOM_STATE, WINDOW_SIZE, TRAIN_RATIO,
    ANOMALY_COLOR, NOMINAL_COLOR,
    LATENT_DIM, HIDDEN_DIM, DROPOUT,
)

tf.keras.utils.set_random_seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

PROCESSED_DIR   = Path('../data/processed/kaggle')
MODELS_DIR      = Path('../models')
SUBMISSIONS_DIR = Path('../submissions')
MODELS_DIR.mkdir(exist_ok=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print('TF', tf.__version__, '- GPUs:', tf.config.list_physical_devices('GPU'))

---
## 1 — Load Data

Preprocessed nominal windows come from NB 02. `X_train_nom` is the full pool
of nominal windows from the training 80%; `X_val` / `y_val` are the row-level
validation 20%.

In [ ]:
with open(PROCESSED_DIR / 'preprocessing_config.json') as f:
    cfg = json.load(f)

SPLIT_IDX = cfg['split_idx']
WIN       = cfg['window_size']
N_FEAT    = cfg['n_features']
assert WIN == WINDOW_SIZE

X_train_nominal = np.load(PROCESSED_DIR / 'X_train_nom.npy')

X_all  = np.load(PROCESSED_DIR / 'train_full_scaled.npy')
y_all  = np.load(PROCESSED_DIR / 'y_train_row.npy')
X_val  = X_all[SPLIT_IDX:]
y_val  = y_all[SPLIT_IDX:]
del X_all, y_all; gc.collect()

X_test   = np.load(PROCESSED_DIR / 'test_scaled.npy')
test_ids = np.load(PROCESSED_DIR / 'test_ids.npy')

val_segments = find_anomaly_segments(y_val)
n_events     = len(val_segments)

print(f'X_train_nominal : {X_train_nominal.shape}')
print(f'X_val           : {X_val.shape}   ({int(y_val.sum()):,} anomalous rows, {n_events} events)')
print(f'X_test          : {X_test.shape}')

---
## 2 — Build and Train LSTM-AE

One LSTM per side, latent 32, dropout 0.2. Each window is z-normalised
per-window before the forward pass (`zscore_window`), so the model
reconstructs *shape* rather than magnitude — this neutralises the scale
drift between train and test that would otherwise dominate a non-linear
reconstruction score.

Training uses a 90/10 split of `X_train_nominal`. The labelled validation
set is never seen during fit.

If `models/lstm_ae.keras` already exists, the cell loads it and skips
training (set `FORCE_RETRAIN = True` to retrain from scratch). A quick
reconstruction-MSE check on the held-out nominal val split confirms the
loaded model is still valid before downstream cells use it.

In [ ]:
def build_lstm_ae(window_size, n_channels,
                  latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT):
    inputs = layers.Input(shape=(window_size, n_channels))
    x      = layers.LSTM(hidden_dim, return_sequences=True, dropout=dropout)(inputs)
    latent = layers.LSTM(latent_dim, return_sequences=False, dropout=dropout)(x)
    x      = layers.RepeatVector(window_size)(latent)
    x      = layers.LSTM(hidden_dim, return_sequences=True, dropout=dropout)(x)
    outputs = layers.TimeDistributed(layers.Dense(n_channels))(x)
    model = Model(inputs, outputs, name='lstm_ae')
    model.compile(optimizer='adam', loss='mse')
    return model


model = build_lstm_ae(WIN, N_FEAT)
model.summary()

In [ ]:
def zscore_window(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6
    return ((X - mu) / sd).astype(np.float32)


rng_split  = np.random.default_rng(RANDOM_STATE)
perm       = rng_split.permutation(len(X_train_nominal))
n_fit      = int(0.9 * len(X_train_nominal))
X_fit      = zscore_window(X_train_nominal[perm[:n_fit]])
X_val_fit  = zscore_window(X_train_nominal[perm[n_fit:]])

EPOCHS        = 25
BATCH_SIZE    = 128
MODEL_PATH    = MODELS_DIR / 'lstm_ae.keras'
FORCE_RETRAIN = False   # set True to retrain even if a saved model exists

history = None
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    print(f'Loading saved model -> {MODEL_PATH}')
    model = tf.keras.models.load_model(MODEL_PATH)
    model.summary()
    val_recon_mse = float(((X_val_fit - model.predict(X_val_fit, batch_size=256, verbose=0)) ** 2).mean())
    print(f'Sanity check — reconstruction MSE on held-out nominal val windows: {val_recon_mse:.6f}')
else:
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
        ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=0),
    ]

    print(f'Training on {len(X_fit):,} nominal windows (fit) / {len(X_val_fit):,} (val) ...')
    t0 = time.time()
    history = model.fit(
        X_fit, X_fit,
        validation_data=(X_val_fit, X_val_fit),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )
    train_seconds = time.time() - t0
    epochs_run    = len(history.history['loss'])
    print(f'Trained in {train_seconds/60:.1f} min ({epochs_run} epochs)')
    print(f'Model saved -> {MODEL_PATH}')

In [ ]:
if history is None:
    print('Skipping training-curve plot — model was loaded from disk, no training history for this run.')
else:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    h = history.history
    ax.plot(h['loss'],     color=NOMINAL_COLOR, lw=1.6, label='Train loss')
    ax.plot(h['val_loss'], color=ANOMALY_COLOR, lw=1.6, label='Val loss (nominal)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.set_yscale('log')
    ax.set_title('LSTM-AE training curves', fontweight='bold')
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()

---
## 3 — Score Validation and Test — Top-k Channel MSE

One scalar per 100-row window, built in two steps: (1) z-normalise the
window, (2) compute per-channel MSE between window and reconstruction,
then take the **mean of the top 5 channels**. This preserves per-channel
anomaly signal that mean-over-all-channels dilutes 58×, since most
ESA-ADB anomalies touch only a handful of telemetry streams.

Val and test windows are built by reshaping row arrays into
non-overlapping windows (stride = WINDOW_SIZE). Trailing rows that don't
fill a full window inherit the last window's score.

In [ ]:
TOPK_CHANNELS = 5

def window_score(model, X, k=TOPK_CHANNELS, batch_size=256):
    Xn           = zscore_window(X)
    Xhat         = model.predict(Xn, batch_size=batch_size, verbose=0)
    per_channel  = ((Xn - Xhat) ** 2).mean(axis=1)
    return np.sort(per_channel, axis=1)[:, -k:].mean(axis=1).astype(np.float32)


def make_windows(X_rows, win=WIN):
    n_full = len(X_rows) // win
    return X_rows[:n_full * win].reshape(n_full, win, X_rows.shape[1]), n_full


X_val_win,  n_val_win  = make_windows(X_val)
X_test_win, n_test_win = make_windows(X_test)

print(f'Scoring val  ({n_val_win:,} windows) ...')
t0 = time.time()
val_scores_win = window_score(model, X_val_win)
print(f'  {time.time()-t0:.1f}s')

print(f'Scoring test ({n_test_win:,} windows) ...')
t0 = time.time()
test_scores = window_score(model, X_test_win)
print(f'  {time.time()-t0:.1f}s')

val_scores = np.repeat(val_scores_win, WIN)
if len(val_scores) < len(X_val):
    val_scores = np.concatenate([val_scores,
                                 np.full(len(X_val) - len(val_scores),
                                         val_scores_win[-1], dtype=np.float32)])

np.save(PROCESSED_DIR / 'lstm_ae_val_scores.npy',  val_scores)
np.save(PROCESSED_DIR / 'lstm_ae_test_scores.npy', test_scores)

print(f'val_scores  : {val_scores.shape}   range [{val_scores.min():.4f}, {val_scores.max():.4f}]')
print(f'test_scores : {test_scores.shape}  range [{test_scores.min():.4f}, {test_scores.max():.4f}]')

---
## 4 — Threshold-Transfer Gate

PCA (NB 04) had a ~900× val-vs-test max ratio and still transferred — so a
large ratio is not an automatic fail. It's a warning sign. We print the
ranges, compute the ratio, and plot both distributions on a shared
log-x axis.

In [ ]:
val_max  = float(val_scores.max())
test_max = float(test_scores.max())
ratio    = val_max / max(test_max, 1e-9)

print(f'Val range  : {val_scores.min():.4f} ... {val_max:.4f}')
print(f'Test range : {test_scores.min():.4f} ... {test_max:.4f}')
print(f'Max ratio (val / test) : {ratio:.1f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

sns.histplot(val_scores[::50],  ax=ax, color=NOMINAL_COLOR, alpha=0.55,
             bins=80, stat='density', log_scale=(True, False),
             label='Val (1:50 sample)')
sns.histplot(test_scores,       ax=ax, color=ANOMALY_COLOR, alpha=0.55,
             bins=80, stat='density', log_scale=(True, False),
             label='Test')
ax.set_xlabel('LSTM-AE window-mean MSE (log scale)')
ax.set_ylabel('Density')
ax.set_title('Score distributions: val vs test', fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

---
## 5 — Threshold Tuning (corrected event-wise F0.5)

200 threshold candidates between the 50th and 99.9th percentile of val scores.
Each candidate turns into a row-level prediction; `corrected_event_f05` scores
it (`Pr_c = Pr_ew · TNR`).

The knee - the lowest threshold within 0.02 F0.5 of the peak - is reported as
a more conservative backup. We use the peak for §8.

In [ ]:
candidates = np.quantile(val_scores, np.linspace(0.5, 0.999, 200))

f05_scores = np.array([
    corrected_event_f05(y_val, (val_scores > t).astype(np.int8))['f_score']
    for t in candidates
], dtype=np.float32)

peak_idx   = int(np.argmax(f05_scores))
peak_t     = float(candidates[peak_idx])
peak_f05   = float(f05_scores[peak_idx])

within = np.where(f05_scores >= peak_f05 - 0.02)[0]
knee_idx = int(within[0])
knee_t   = float(candidates[knee_idx])
knee_f05 = float(f05_scores[knee_idx])

best_t   = peak_t
best_f05 = peak_f05

print(f'Peak  : threshold = {peak_t:.6f}   F0.5 = {peak_f05:.4f}')
print(f'Knee  : threshold = {knee_t:.6f}   F0.5 = {knee_f05:.4f}  (backup)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(candidates, f05_scores, color='#8e44ad', lw=1.6)
ax.axvline(peak_t, color='black', ls='--', lw=1,
           label=f'Peak  F0.5={peak_f05:.3f}  t={peak_t:.4f}')
ax.axvline(knee_t, color='#e67e22', ls=':', lw=1,
           label=f'Knee  F0.5={knee_f05:.3f}  t={knee_t:.4f}')
ax.set_xlabel('Threshold (val-score quantile)')
ax.set_ylabel('Corrected event-wise F0.5')
ax.set_title('F0.5 vs threshold', fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

The raw F0.5-peak threshold assumes val and test scores share a magnitude
scale. When the transfer gate in §4 shows that they don't, we fall back to the
matching val quantile applied to the test score distribution.

In [ ]:
def apply_threshold_fallback(best_t, val_scores, test_scores):
    if test_scores.min() > best_t or test_scores.max() < best_t:
        q = float((val_scores <= best_t).mean())
        return float(np.quantile(test_scores, q)), q
    return float(best_t), None


applied_t, fallback_q = apply_threshold_fallback(best_t, val_scores, test_scores)
if fallback_q is not None:
    print(f'Transfer gate failed — percentile fallback at q={fallback_q:.4f}, t={applied_t:.4f}')
else:
    print(f'Raw threshold transferred — using t={applied_t:.4f}')

expected_rate = float((test_scores > applied_t).mean())
print(f'Expected test positive rate: {expected_rate:.2%}')

---
## 6 — Validation Timeline and Per-Event Analysis

Where does the model actually flag anomalies in the val timeline, and which
of the 38 ground-truth events does it hit?

In [ ]:
y_pred_val = (val_scores > best_t).astype(np.int8)

metric = corrected_event_f05(y_val, y_pred_val)
tp_events      = metric['tp_events']
fn_events      = metric['fn_events']
fp_pred_events = metric['fp_pred_events']
tnr            = metric['tnr']
precision      = metric['precision']
recall         = metric['recall']

n_detected = tp_events
n_missed   = fn_events

print(f'Val F0.5            : {metric["f_score"]:.4f}')
print(f'Precision (Pr_c)    : {precision:.4f}')
print(f'Recall (event-wise) : {recall:.4f}')
print(f'TNR                 : {tnr:.4f}')
print(f'TP events           : {tp_events} / {n_events}')
print(f'FN events (missed)  : {n_missed}')
print(f'FP predicted events : {fp_pred_events}')

In [ ]:
DS = 50
val_idx_ds = np.arange(0, len(val_scores), DS)
scores_ds  = val_scores[val_idx_ds]
y_val_ds   = y_val[val_idx_ds]

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.plot(val_idx_ds, scores_ds, lw=0.4, color='#7f8c8d', alpha=0.75)
ax.axhline(best_t, color=ANOMALY_COLOR, lw=1.4, ls='--',
           label=f'Threshold = {best_t:.4f}')
for seg in val_segments:
    ax.axvspan(seg['start'], seg['end'], color=ANOMALY_COLOR, alpha=0.15, linewidth=0)
ax.set_ylabel('LSTM-AE MSE')
ax.set_title('LSTM-AE window-mean MSE over validation timeline', fontweight='bold')
handles = [
    plt.Line2D([0],[0], color='#7f8c8d', lw=1, label='Recon MSE'),
    plt.Line2D([0],[0], color=ANOMALY_COLOR, lw=1.4, ls='--', label=f'Threshold {best_t:.4f}'),
    mpatches.Patch(color=ANOMALY_COLOR, alpha=0.3, label='True anomaly region'),
]
ax.legend(handles=handles, fontsize=8, loc='upper right')

ax2 = axes[1]
pred_ds = (scores_ds > best_t).astype(np.float32)
ax2.fill_between(val_idx_ds, pred_ds, color='#8e44ad', alpha=0.6, label='Predicted')
ax2.fill_between(val_idx_ds, y_val_ds.astype(np.float32),
                 color=ANOMALY_COLOR, alpha=0.4, label='True')
ax2.set_yticks([0, 1])
ax2.set_ylabel('Anomaly')
ax2.set_xlabel('Val row index')
ax2.legend(fontsize=8, loc='upper right')
fig.tight_layout()
plt.show()

In [ ]:
event_rows = []
for seg in val_segments:
    pred_in_seg = y_pred_val[seg['start']:seg['end']+1]
    event_rows.append({
        'start'   : seg['start'],
        'end'     : seg['end'],
        'length'  : seg['length'],
        'detected': bool(pred_in_seg.any()),
        'hit_rate': float(pred_in_seg.mean()),
    })
event_df = pd.DataFrame(event_rows)

missed = event_df[~event_df['detected']]
print(f'Events detected : {int(event_df["detected"].sum())} / {n_events}')
print(f'Events missed   : {len(missed)}')
if len(missed):
    print('\nMissed events:')
    print(missed[['start','end','length']].to_string(index=False))

---
## 7 — Sensitivity: latent_dim and hidden_dim

Quick exploration on a 10 % subsample of `X_train_nominal` (every 10th window)
— enough to rank configurations, not enough to retrain the canonical model.
The §2 model remains the one we use downstream.

In [ ]:
SKIP_SENSITIVITY = True   # set to False to run the latent/hidden sweep

if SKIP_SENSITIVITY:
    print('Sensitivity sweep skipped (set SKIP_SENSITIVITY = False to run).')
else:
    X_proto   = zscore_window(X_train_nominal[::10])
    SUB_EPOCHS = 10

    def quick_train_and_score(latent_dim, hidden_dim):
        tf.keras.utils.set_random_seed(RANDOM_STATE)
        m = build_lstm_ae(WIN, N_FEAT, latent_dim=latent_dim, hidden_dim=hidden_dim)
        m.fit(X_proto, X_proto,
              validation_split=0.1,
              epochs=SUB_EPOCHS, batch_size=BATCH_SIZE,
              callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
              verbose=0, shuffle=True)
        s_win = window_score(m, X_val_win)
        s_row = np.repeat(s_win, WIN)
        if len(s_row) < len(y_val):
            s_row = np.concatenate([s_row, np.full(len(y_val) - len(s_row), s_win[-1], dtype=np.float32)])
        cand  = np.quantile(s_row, np.linspace(0.5, 0.999, 100))
        f05   = max(corrected_event_f05(y_val, (s_row > t).astype(np.int8))['f_score'] for t in cand)
        tf.keras.backend.clear_session()
        return f05

    sweep = []
    for l in [16, 32, 64]:
        f = quick_train_and_score(latent_dim=l, hidden_dim=HIDDEN_DIM)
        sweep.append({'sweep': 'latent_dim', 'latent_dim': l, 'hidden_dim': HIDDEN_DIM, 'val_f05': f})
        print(f'  latent_dim={l:3d}  hidden_dim={HIDDEN_DIM}   val F0.5 = {f:.4f}')

    for h in [32, 64, 128]:
        f = quick_train_and_score(latent_dim=LATENT_DIM, hidden_dim=h)
        sweep.append({'sweep': 'hidden_dim', 'latent_dim': LATENT_DIM, 'hidden_dim': h, 'val_f05': f})
        print(f'  latent_dim={LATENT_DIM}  hidden_dim={h:3d}   val F0.5 = {f:.4f}')

    sens_df = pd.DataFrame(sweep)
    print()
    print(sens_df.round(4).to_string(index=False))

---
## 8 — Generate Test Predictions and Save Submission

Window-level predictions from the val-tuned threshold, expanded to row level
by repeating each window's prediction WINDOW_SIZE times. Trailing test rows
that don't fill a full window inherit the last window's prediction.

In [ ]:
test_preds_window = (test_scores > applied_t).astype(np.uint8)
test_preds_row    = np.repeat(test_preds_window, WIN)

test_df = pd.read_parquet('../data/raw/test.parquet', columns=['id'])
if len(test_preds_row) < len(test_df):
    pad = np.full(len(test_df) - len(test_preds_row),
                  test_preds_window[-1], dtype=np.uint8)
    test_preds_row = np.concatenate([test_preds_row, pad])
test_preds_row = test_preds_row[:len(test_df)]

submission = pd.DataFrame({'id': test_df['id'], 'is_anomaly': test_preds_row})
assert list(submission['id']) == list(test_ids), 'ID mismatch vs processed test_ids'
submission.to_parquet(SUBMISSIONS_DIR / 'baseline_lstm_ae.parquet', index=False)

n_seg = int(np.sum(np.diff(np.concatenate([[0], test_preds_row, [0]])) == 1))
pos_rate = float(test_preds_row.mean())
print(f'Submission saved -> submissions/baseline_lstm_ae.parquet')
print(f'Positive rate        : {pos_rate:.3%}')
print(f'Predicted segments   : {n_seg}')
if not (0.02 <= pos_rate <= 0.15):
    print(f'  WARNING: positive rate outside healthy range [2%, 15%]')
if not (10 <= n_seg <= 500):
    print(f'  WARNING: segment count outside healthy range [10, 500]')

---
## 9 — Compare to PCA Baseline

PCA is refit here on the flattened nominal windows (`n_components=0.95`) so
the comparison uses consistent code paths with the ensemble in §10 — no
dependency on cached NB 04 artefacts. Both baselines run the same
threshold-tuning + fallback pipeline.

In [ ]:
from scipy.stats import rankdata
from sklearn.decomposition import PCA

X_train_flat = X_train_nominal.reshape(len(X_train_nominal), -1)
X_val_flat   = X_val_win.reshape(n_val_win, -1)
X_test_flat  = X_test_win.reshape(n_test_win, -1)

pca_model    = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(X_train_flat)
pca_val_win  = ((X_val_flat  - pca_model.inverse_transform(pca_model.transform(X_val_flat)))  ** 2).mean(axis=1).astype(np.float32)
pca_test_win = ((X_test_flat - pca_model.inverse_transform(pca_model.transform(X_test_flat))) ** 2).mean(axis=1).astype(np.float32)

pca_val_scores = np.repeat(pca_val_win, WIN)
if len(pca_val_scores) < len(X_val):
    pca_val_scores = np.concatenate([pca_val_scores,
                                     np.full(len(X_val) - len(pca_val_scores),
                                             pca_val_win[-1], dtype=np.float32)])
pca_test_scores = pca_test_win

print(f'PCA components kept   : {pca_model.n_components_}')
print(f'pca_val_scores range  : [{pca_val_scores.min():.4f}, {pca_val_scores.max():.4f}]')
print(f'pca_test_scores range : [{pca_test_scores.min():.4f}, {pca_test_scores.max():.4f}]')

In [ ]:
pca_cand     = np.quantile(pca_val_scores, np.linspace(0.5, 0.999, 200))
pca_f05      = np.array([
    corrected_event_f05(y_val, (pca_val_scores > t).astype(np.int8))['f_score']
    for t in pca_cand
], dtype=np.float32)
pca_best_t   = float(pca_cand[int(np.argmax(pca_f05))])
pca_best_f05 = float(pca_f05.max())

pca_applied_t, pca_fallback_q = apply_threshold_fallback(
    pca_best_t, pca_val_scores, pca_test_scores
)
pca_thr_source = 'raw' if pca_fallback_q is None else 'fallback'

pca_val_pred      = (pca_val_scores > pca_best_t).astype(np.int8)
pca_val_metric    = corrected_event_f05(y_val, pca_val_pred)

pca_test_pred_win = (pca_test_scores > pca_applied_t).astype(np.uint8)
pca_test_pred_row = np.repeat(pca_test_pred_win, WIN)
if len(pca_test_pred_row) < len(test_df):
    pca_test_pred_row = np.concatenate([pca_test_pred_row,
                                        np.full(len(test_df) - len(pca_test_pred_row),
                                                pca_test_pred_win[-1], dtype=np.uint8)])
pca_test_pred_row = pca_test_pred_row[:len(test_df)]

pca_test_rate = float(pca_test_pred_row.mean())
pca_test_segs = int(np.sum(np.diff(np.concatenate([[0], pca_test_pred_row, [0]])) == 1))
pca_ratio     = float(pca_val_scores.max()) / max(float(pca_test_scores.max()), 1e-9)

print(f'PCA peak F0.5 : {pca_best_f05:.4f}  (threshold {pca_best_t:.6f}, applied {pca_applied_t:.6f}, {pca_thr_source})')

In [ ]:
val_scores_ens  = 0.5 * rankdata(val_scores)  / len(val_scores)  + 0.5 * rankdata(pca_val_scores)  / len(pca_val_scores)
test_scores_ens = 0.5 * rankdata(test_scores) / len(test_scores) + 0.5 * rankdata(pca_test_scores) / len(pca_test_scores)

candidates_ens = np.quantile(val_scores_ens, np.linspace(0.5, 0.999, 200))
f05_ens        = np.array([
    corrected_event_f05(y_val, (val_scores_ens > t).astype(np.int8))['f_score']
    for t in candidates_ens
], dtype=np.float32)
best_t_ens   = float(candidates_ens[int(np.argmax(f05_ens))])
best_f05_ens = float(f05_ens.max())

applied_t_ens, fallback_q_ens = apply_threshold_fallback(
    best_t_ens, val_scores_ens, test_scores_ens
)
ens_thr_source = 'raw' if fallback_q_ens is None else 'fallback'

ens_val_pred      = (val_scores_ens > best_t_ens).astype(np.int8)
ens_val_metric    = corrected_event_f05(y_val, ens_val_pred)

ens_ratio = float(val_scores_ens.max()) / max(float(test_scores_ens.max()), 1e-9)
print(f'Ensemble peak F0.5 : {best_f05_ens:.4f}  (threshold {best_t_ens:.4f}, applied {applied_t_ens:.4f}, {ens_thr_source})')

In [ ]:
lstm_thr_source = 'raw' if fallback_q is None else 'fallback'

cmp = pd.DataFrame({
    'Metric': [
        'Val F0.5 (peak)',
        'Val events detected / 38',
        'Val FP predicted segments',
        'Test positive rate',
        'Test predicted segments',
        'Val / test max-score ratio',
        'Threshold source',
    ],
    'PCA': [
        f'{pca_best_f05:.4f}',
        f'{pca_val_metric["tp_events"]}',
        f'{pca_val_metric["fp_pred_events"]}',
        f'{pca_test_rate:.3%}',
        f'{pca_test_segs}',
        f'{pca_ratio:.1f}x',
        pca_thr_source,
    ],
    'LSTM-AE': [
        f'{peak_f05:.4f}',
        f'{tp_events}',
        f'{fp_pred_events}',
        f'{pos_rate:.3%}',
        f'{n_seg}',
        f'{ratio:.1f}x',
        lstm_thr_source,
    ],
    'Ensemble': [
        f'{best_f05_ens:.4f}',
        f'{ens_val_metric["tp_events"]}',
        f'{ens_val_metric["fp_pred_events"]}',
        '(see §10)',
        '(see §10)',
        f'{ens_ratio:.1f}x',
        ens_thr_source,
    ],
})
print(cmp.to_string(index=False))

---
## 10 — Ensemble with PCA Baseline (rank-averaged)

The LSTM-AE and PCA score distributions live on different scales, so we rank-fuse
them (50/50) before thresholding. The ensemble threshold is tuned on val with the
same corrected F0.5 pipeline and the percentile-matched fallback from §5, and the
resulting submission is written to `submissions/baseline_ensemble.parquet`.

In [ ]:
ens_test_pred_win = (test_scores_ens > applied_t_ens).astype(np.uint8)
ens_test_pred_row = np.repeat(ens_test_pred_win, WIN)
if len(ens_test_pred_row) < len(test_df):
    ens_test_pred_row = np.concatenate([ens_test_pred_row,
                                        np.full(len(test_df) - len(ens_test_pred_row),
                                                ens_test_pred_win[-1], dtype=np.uint8)])
ens_test_pred_row = ens_test_pred_row[:len(test_df)]

ens_submission = pd.DataFrame({'id': test_df['id'], 'is_anomaly': ens_test_pred_row})
assert list(ens_submission['id']) == list(test_ids), 'ID mismatch vs processed test_ids'
ens_submission.to_parquet(SUBMISSIONS_DIR / 'baseline_ensemble.parquet', index=False)

ens_pos_rate = float(ens_test_pred_row.mean())
ens_n_seg    = int(np.sum(np.diff(np.concatenate([[0], ens_test_pred_row, [0]])) == 1))
print(f'Ensemble submission saved -> submissions/baseline_ensemble.parquet')
print(f'Ensemble positive rate      : {ens_pos_rate:.3%}')
print(f'Ensemble predicted segments : {ens_n_seg}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

sns.histplot(val_scores_ens[::50], ax=ax, color='#8e44ad', alpha=0.5, bins=80,
             stat='density', label='Ensemble val (1:50)')
sns.histplot(val_scores[::50],     ax=ax, color=NOMINAL_COLOR, alpha=0.45, bins=80,
             stat='density', label='LSTM-AE val (1:50)')
sns.histplot(pca_val_scores[::50], ax=ax, color=ANOMALY_COLOR, alpha=0.45, bins=80,
             stat='density', label='PCA val (1:50)')
ax.set_yscale('log')
ax.set_xlabel('Score (rank-normalised for ensemble; raw for individual baselines)')
ax.set_ylabel('Density (log)')
ax.set_title('Val score distributions: LSTM-AE vs PCA vs ensemble', fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(candidates_ens, f05_ens, color='#16a085', lw=1.6)
ax.axvline(best_t_ens, color='black', ls='--', lw=1,
           label=f'Peak  F0.5={best_f05_ens:.3f}  t={best_t_ens:.4f}')
ax.set_xlabel('Ensemble threshold (val-score quantile)')
ax.set_ylabel('Corrected event-wise F0.5')
ax.set_title('Ensemble F0.5 vs threshold', fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

---
## Summary

| Model | Val F0.5 | Kaggle public | Kaggle private | Test pos-rate | Threshold |
|---|---|---|---|---|---|
| Isolation Forest | 0.091 | — | — | 0.00% | raw (collapsed) |
| PCA (k=39, windowed) | **0.770** | 0.522 | **0.599** | 6.35% | raw |
| PCA (row-level) | 0.119 | 0.522 | 0.294 | — | raw |
| LSTM-AE v1 (raw windows, mean over all axes) | 0.736 | 0.112 | 0.078 | 17.9% | fallback |
| LSTM-AE v2 (z-norm + top-k=5 channel) | 0.324 | _pending_ | _pending_ | 0.04% | raw |
| Ensemble v1 (PCA + LSTM-AE v1) | 0.684 | 0.522 | 0.476 | 6.3% | raw |
| Ensemble v2 (PCA + LSTM-AE v2) | 0.667 | _pending_ | _pending_ | 0.15% | raw |

PCA wins on private. Three submissions tie on public at 0.522 but diverge
on private (0.599 / 0.476 / 0.294) — public-leaderboard overfitting is
real, and the row-level PCA variant is the cautionary case: same public,
private halves. NB 04's 5-fold temporal CV (mean 0.641 ± 0.313) shows why
the 0.770 val score transfers so cleanly on PCA: one fold collapses on a
magnitude-outlier event, the other three sit at 0.76–0.82.

### Why vanilla LSTM-AE loses to PCA
A reconstruction autoencoder scores distance from the training manifold,
which equals anomaly likelihood only under stationary nominal. ESA-ADB has
0.43 mean KS drift between train and test (NB 01). PCA's linear projection
tolerates that drift; the ~50k-parameter LSTM overfits the narrow training
nominal and loses contrast on later-mission nominal windows — the 407×
val/test max ratio in §4 is the direct symptom.

### v2 changes (applied in §2 / §3)
- **Per-window z-normalisation** before the forward pass: the model
  reconstructs shape, not magnitude — neutralises scale drift.
- **Top-k (k=5) channel aggregation** at scoring: preserves per-channel
  anomaly signal that mean-over-58 dilutes.

**What v2 actually did**: the drift problem is structurally fixed —
val/test max ratio collapsed from 407× to 1.1×, and the raw threshold
transfers without a fallback. But recall collapsed too: val F0.5 fell to
0.324, with only 10 of 38 events detected and 2 segments flagged on
test. v1 overflagged (TNR collapse on Kaggle); v2 underflags (recall
collapse). A vanilla reconstruction LSTM-AE on ESA-ADB cannot be
simultaneously drift-robust *and* anomaly-sensitive without
domain-specific changes.

### Ceiling
Events shorter than WINDOW_SIZE are smeared by windowing regardless of
model; temporal drift limits any val-tuned threshold.

### Next
Overlapping scoring windows (stride < WINDOW_SIZE); include val-nominal in
training; anomaly-rate prior instead of val-peak quantile in the fallback;
positive-rate plausibility gate in model selection.